In [ ]:


import os
from google.cloud import bigquery, storage



BUCKET = os.environ["WORKSPACE_BUCKET"].replace("gs://", "").strip("/")
CDR    = os.environ["WORKSPACE_CDR"]

bq  = bigquery.Client()
gcs = storage.Client()

print(f"Bucket : gs://{BUCKET}")
print(f"CDR    : {CDR}")


DATA_PREFIX = "sjl/data"
bucket_obj  = gcs.bucket(BUCKET)
stale_blobs = list(bucket_obj.list_blobs(prefix=DATA_PREFIX + "/"))

if stale_blobs:
    bucket_obj.delete_blobs(stale_blobs)
    print(f"Deleted {len(stale_blobs):,} stale blobs from gs://{BUCKET}/{DATA_PREFIX}/")
else:
    print("No previous data found — clean slate ready.")


def export(query: str, shard_glob: str) -> None:
    """
    Run a BigQuery EXPORT DATA statement writing Parquet shards to GCS.

    Parameters
    ----------
    query      : SELECT statement to export (no trailing semicolon).
    shard_glob : Path relative to bucket root, must end in _*.parquet.
    """
    uri = f"gs://{BUCKET}/{shard_glob}"
    job = bq.query(
        f"EXPORT DATA OPTIONS("
        f"  uri='{uri}', format='PARQUET', overwrite=true"
        f") AS\n{query}"
    )
    job.result()   
    print(f"  ✓  {uri}")


print("\nExporting person...")
export(
    f"""
    SELECT
        person.person_id,
        person.year_of_birth,
        concept.concept_name AS sex_at_birth
    FROM `{CDR}.person` AS person
    LEFT JOIN `{CDR}.concept` AS concept
        ON person.sex_at_birth_concept_id = concept.concept_id
    """,
    f"{DATA_PREFIX}/person_*.parquet",
)



print("Exporting survey...")

SURVEY_CONCEPT_IDS = ", ".join(str(i) for i in (
    1586140,   # Race/Ethnicity
    1585940,   # Education Level: Highest Grade
    1585857,   # Income: Annual Income
    1586198,   # Employment Status: Employment Status
    1586201,   # Employment Status: Current Employment
    1585952,   # Alcohol (participant + frequency)
    1585375,   # Smoking: 100 Cigs Lifetime
))

export(
    f"""
    SELECT
        person_id,
        survey_datetime,
        question_concept_id,
        question,
        answer
    FROM `{CDR}.ds_survey`
    WHERE question_concept_id IN ({SURVEY_CONCEPT_IDS})
    """,
    f"{DATA_PREFIX}/survey_*.parquet",
)


print("Exporting sleep levels...")
export(
    f"""
    SELECT
        person_id,
        sleep_date,
        is_main_sleep,
        level,
        start_datetime,
        duration_in_min
    FROM `{CDR}.sleep_level`
    WHERE is_main_sleep = 'true'
    """,
    f"{DATA_PREFIX}/sleepmin_*.parquet",
)


print("Exporting daily steps...")
export(
    f"""
    SELECT
        person_id,
        date,
        steps
    FROM `{CDR}.activity_summary`
    """,
    f"{DATA_PREFIX}/steps_*.parquet",
)

print("Exporting deprivation index...")
export(
    f"""
    SELECT
        PERSON_ID         AS person_id,
        DEPRIVATION_INDEX AS deprivation_index
    FROM `{CDR}.ds_zip_code_socioeconomic`
    """,
    f"{DATA_PREFIX}/deprivation_*.parquet",
)

print("Exporting BMI...")
export(
    f"""
    SELECT
        PERSON_ID                          AS person_id,
        CAST(MEASUREMENT_DATETIME AS DATE) AS measurement_date,
        VALUE_AS_NUMBER                    AS bmi,
        CASE
            WHEN MEASUREMENT_SOURCE_CONCEPT_ID = 903124 THEN 'AoU'
            ELSE 'EHR'
        END                                AS source
    FROM `{CDR}.ds_measurement`
    WHERE
        (MEASUREMENT_SOURCE_CONCEPT_ID = 903124
         OR MEASUREMENT_CONCEPT_ID = 3038553)
        AND VALUE_AS_NUMBER BETWEEN 10 AND 150
    """,
    f"{DATA_PREFIX}/bmi_*.parquet",
)


print("Exporting EHR length...")
export(
    f"""
    SELECT
        person_id,
        DATE_DIFF(
            MAX(measurement_date),
            MIN(measurement_date),
            DAY
        ) / 365.25            AS ehr_length_years,
        MAX(measurement_date) AS last_observation_date
    FROM `{CDR}.measurement`
    GROUP BY person_id
    """,
    f"{DATA_PREFIX}/ehr_length_*.parquet",
)


print("\n=== DATA PULL COMPLETE ===")
print(f"All tables written to gs://{BUCKET}/{DATA_PREFIX}/")
print("\nNext: run munge_data.py (Notebook 1)")